In [1]:
import os
print(os.getcwd())
os.chdir('/home/user1/MXY/EHRScore') # Change to the project directory

/home/user1/MXY/EHRScore/data/mimic4


In [2]:
import pandas as pd

# read formatted mimic4 data
os.makedirs("data/mimic4/preprocessed", exist_ok=True)
data_path = "data/mimic4/format_mimic4_anonymized.csv"
mimic4_df = pd.read_csv(data_path)
mimic4_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10001,1,1,2111-06-04 22:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,2809;2731;4019;2113;53560;5533;56210;4240;V103,"Iron deficiency anemia, unspecified;Monoclonal...",Ferric Gluconate;Pneumococcal Vac Polyvalent;S...,4516;4542,Esophagogastroduodenoscopy [EGD] with closed b...
1,10001,1,2,2111-06-05 10:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10001,1,3,2111-06-05 22:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10001,1,4,2111-06-06 10:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,1,5,2111-06-06 22:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4082066,190677,3,15,2142-02-11 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,36.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4082067,190677,3,16,2142-02-12 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,36.83,98.0,NaN,NaN,7.34,NaN,NaN,NaN,NaN,NaN
4082068,190677,3,17,2142-02-12 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,35.40,96.0,NaN,NaN,7.23,NaN,NaN,NaN,NaN,NaN
4082069,190677,3,18,2142-02-13 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,35.70,97.0,NaN,75.0,7.23,NaN,NaN,NaN,NaN,NaN


In [3]:
import numpy as np

TASK                    = "outcome"        # "outcome" or "readmission"
RNG_SEED                = 42

DESIRED_TOTAL           = 20_000               # outcome: mimci4:20_000  readmission: mimic4:20_000

# Outcome
DESIRED_POS_RATIO_OUTCOME = 0.13              

# Readmission
TARGET_RATE_READM       = 0.17                
TOLERANCE_READM         = 0.005               
# ==========================================


filtered_df = (
    mimic4_df.groupby("PatientID")
             .filter(lambda x: len(x) >= 10)
)
print(f"[INFO] {filtered_df['PatientID'].nunique():,} patients remaining after filtering")

rng = np.random.default_rng(RNG_SEED)

# Outcome TASK

if TASK.lower() == "outcome":
    label_col = "Outcome"

    patient_label_df = (
        filtered_df.groupby("PatientID")[label_col]
                   .max()
                   .reset_index()
    )
    pos_ids = patient_label_df.query(f"{label_col} == 1")["PatientID"].to_numpy()
    neg_ids = patient_label_df.query(f"{label_col} == 0")["PatientID"].to_numpy()

    desired_pos = int(DESIRED_TOTAL * DESIRED_POS_RATIO_OUTCOME)
    desired_neg = DESIRED_TOTAL - desired_pos

    sampled_pos = rng.choice(pos_ids, size=min(desired_pos, len(pos_ids)), replace=False)
    sampled_neg = rng.choice(neg_ids, size=min(desired_neg, len(neg_ids)), replace=False)

    if len(sampled_pos) < desired_pos:
        need = desired_pos - len(sampled_pos)
        extra_neg = rng.choice(
            [i for i in neg_ids if i not in sampled_neg], size=need, replace=False
        )
        sampled_neg = np.concatenate([sampled_neg, extra_neg])

    if len(sampled_neg) < desired_neg:
        need = desired_neg - len(sampled_neg)
        extra_pos = rng.choice(
            [i for i in pos_ids if i not in sampled_pos], size=need, replace=False
        )
        sampled_pos = np.concatenate([sampled_pos, extra_pos])

    selected_patients = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(selected_patients)

    print(
        f"[Outcome] Selected {len(selected_patients):,} patients "
        f"(Positive: {len(sampled_pos):,}, Negative: {len(sampled_neg):,})"
    )

    mimic4_filtered_df = (
        filtered_df[filtered_df["PatientID"].isin(selected_patients)]
        .groupby(["PatientID", "VisitID"])
        .head(48)
        .reset_index(drop=True)
    )



# Readmission TASK

elif TASK.lower() == "readmission":
    readmit_col = "Readmission"
    visit_labels   = {}  
    patient_labels = {}  
    processed_parts = []

    for pid, grp in filtered_df.groupby("PatientID"):
        grp = grp.sort_values("VisitID")        
        visits = grp["VisitID"].unique()

        visit_flag = (
            grp.groupby("VisitID")[readmit_col]
               .max()
               .reindex(visits, fill_value=0)
               .to_numpy()
        )

        last_one_idx = np.where(visit_flag == 1)[0][-1] if visit_flag.any() else None
        keep_visits  = visits if last_one_idx is None else visits[: last_one_idx + 1]

        has_positive = False
        for vid, flag in zip(visits, visit_flag):
            if vid in keep_visits:
                visit_labels[(pid, vid)] = int(flag)
                if flag == 1:
                    has_positive = True
        patient_labels[pid] = int(has_positive)

        processed_parts.append(grp[grp["VisitID"].isin(keep_visits)])

    processed_df = pd.concat(processed_parts, ignore_index=True)

    pos_pids = [pid for pid, lab in patient_labels.items() if lab == 1]
    neg_pids = [pid for pid, lab in patient_labels.items() if lab == 0]

    desired_pos = int(DESIRED_TOTAL * TARGET_RATE_READM)
    desired_neg = DESIRED_TOTAL - desired_pos

    sampled_pos = rng.choice(pos_pids, size=min(desired_pos, len(pos_pids)), replace=False)
    sampled_neg = rng.choice(neg_pids, size=min(desired_neg, len(neg_pids)), replace=False)

    if len(sampled_pos) < desired_pos and len(neg_pids) > len(sampled_neg):
        need = desired_pos - len(sampled_pos)
        extra_neg = rng.choice(
            [i for i in neg_pids if i not in sampled_neg],
            size=min(need, len(neg_pids) - len(sampled_neg)),
            replace=False,
        )
        sampled_neg = np.concatenate([sampled_neg, extra_neg])

    if len(sampled_neg) < desired_neg and len(pos_pids) > len(sampled_pos):
        need = desired_neg - len(sampled_neg)
        extra_pos = rng.choice(
            [i for i in pos_pids if i not in sampled_pos],
            size=min(need, len(pos_pids) - len(sampled_pos)),
            replace=False,
        )
        sampled_pos = np.concatenate([sampled_pos, extra_pos])

    selected_pids = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(selected_pids)

    actual_rate = len(sampled_pos) / len(selected_pids)
    print(
        f"[Readmission] Selected {len(selected_pids):,} patients, "
        f"Readmission rate: {actual_rate:.3%} (Target: {TARGET_RATE_READM:.3%})"
    )

    mimic4_filtered_df = (
        processed_df[processed_df["PatientID"].isin(selected_pids)]
        .groupby(["PatientID", "VisitID"])
        .head(48)
        .reset_index(drop=True)
    )



[INFO] 93,461 patients remaining after filtering
[Outcome] Selected 20,000 patients (Positive: 2,600, Negative: 17,400)


In [4]:
# Perform LOCF(Last Observation Carried Forward) imputation
exclude_cols = [
    'Conditions_ICD9',
    'Conditions_Long',
    'Medications',
    'Procedures_ICD9',
    'Procedures_Long'
]

cols_to_impute = [col for col in mimic4_filtered_df.columns if col not in exclude_cols]

# locf
locf_imputed_df = mimic4_filtered_df.copy()
locf_imputed_df[cols_to_impute] = locf_imputed_df.groupby(['PatientID', 'VisitID'])[cols_to_impute].ffill()

# nocb
nocb_imputed_df = locf_imputed_df.copy()
nocb_imputed_df[cols_to_impute] = nocb_imputed_df.groupby(['PatientID', 'VisitID'])[cols_to_impute].bfill()

nocb_imputed_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10002,1,1,2122-01-19 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,71595;2851;4580;2724;V1272,"Osteoarthrosis, unspecified whether generalize...",Enoxaparin Sodium;0.9% Sodium Chloride;CefazoL...,8151,Total hip replacement
1,10002,1,2,2122-01-19 19:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10002,1,3,2122-01-20 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10002,1,4,2122-01-20 19:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10002,1,5,2122-01-21 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
781389,190677,3,15,2142-02-11 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,36.89,98.0,NaN,128.0,7.41,NaN,NaN,NaN,NaN,NaN
781390,190677,3,16,2142-02-12 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,36.83,98.0,NaN,128.0,7.34,NaN,NaN,NaN,NaN,NaN
781391,190677,3,17,2142-02-12 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,35.40,96.0,NaN,128.0,7.23,NaN,NaN,NaN,NaN,NaN
781392,190677,3,18,2142-02-13 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,35.70,97.0,NaN,75.0,7.23,NaN,NaN,NaN,NaN,NaN


In [5]:
# Save the time-series data for subsequent statistical analysis
timeseries_df = nocb_imputed_df[cols_to_impute]
timeseries_df.to_csv("data/mimic4/preprocessed/timeseries_mimic4.csv", index=False)
timeseries_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Diastolic blood pressure,Systolic blood pressure,Mean blood pressure,Heart Rate,Respiratory rate,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH
0,10002,1,1,2122-01-19 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10002,1,2,2122-01-19 19:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10002,1,3,2122-01-20 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10002,1,4,2122-01-20 19:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10002,1,5,2122-01-21 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
781389,190677,3,15,2142-02-11 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,53.0,80.0,64.0,120.0,26.0,36.89,98.0,NaN,128.0,7.41
781390,190677,3,16,2142-02-12 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,60.0,101.0,78.0,118.0,26.0,36.83,98.0,NaN,128.0,7.34
781391,190677,3,17,2142-02-12 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,53.0,93.0,69.0,125.0,20.0,35.40,96.0,NaN,128.0,7.23
781392,190677,3,18,2142-02-13 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,50.0,87.0,64.0,124.0,21.0,35.70,97.0,NaN,75.0,7.23


In [6]:
# Extract the patient's conditions, medications, and procedures to CMPs.csv
clinical_cols = [
    'PatientID', 'VisitID',
    'AdmissionTime', 'DischargeTime',
    'Conditions_ICD9', 'Conditions_Long',
    'Medications',
    'Procedures_ICD9', 'Procedures_Long'
]
cmps_df = nocb_imputed_df.groupby(['PatientID', 'VisitID']).first().reset_index()[clinical_cols]

cmps_df.to_csv("data/mimic4/preprocessed/CMPs_mimic4.csv", index=False)
cmps_df

,PatientID,VisitID,AdmissionTime,DischargeTime,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10002,1,2122-01-19 07:15:00,2122-01-24 12:10:00,71595;2851;4580;2724;V1272,"Osteoarthrosis, unspecified whether generalize...",Enoxaparin Sodium;0.9% Sodium Chloride;CefazoL...,8151,Total hip replacement
1,10045,1,2160-09-11 00:54:00,2160-09-17 18:04:00,4465;5849;514;4373;37852;V8542;25062;3572;2852...,"Giant cell arteritis;Acute kidney failure, uns...",Sertraline;Vitamin D;Insulin;Potassium Chlorid...,3821,Biopsy of blood vessel
2,10045,2,2161-01-16 16:11:00,2161-01-30 18:50:00,27803;42833;5849;V8541;25080;2859;4280;311;530...,Obesity hypoventilation syndrome;Acute on chro...,Insulin;Furosemide;Furosemide;Ondansetron;Insu...,3721;0059,Right heart cardiac catheterization;Intravascu...
3,10051,1,2164-10-29 22:23:00,2164-11-03 17:20:00,None,None,PNEUMOcoccal 23-valent polysaccharide vaccine;...,None,None
4,10058,1,2160-01-24 22:29:00,2160-01-30 15:00:00,431;43883;30390;4019;42789;3051;V1254;2724,Intracerebral hemorrhage;Other late effects of...,Lisinopril;Sodium Chloride 0.9% Flush;Multivi...,9604,Insertion of endotracheal tube
...,...,...,...,...,...,...,...,...,...
72234,190653,2,2188-03-16 16:49:00,2188-03-19 18:30:00,27549;5849;71225;71535;71906;34590;4019;53081;...,Other disorders of calcium metabolism;Acute ki...,Valproic Acid;SW;Magnesium Sulfate;Pneumococca...,8191,Arthrocentesis
72235,190674,1,2139-01-20 16:49:00,2139-01-26 13:44:00,5070;5990;3320;2731;04104;42731;496;3331;48283...,Pneumonitis due to inhalation of food or vomit...,Influenza Vaccine Quadrivalent;Potassium Chl 2...,None,None
72236,190677,1,2140-04-01 00:16:00,2140-04-12 13:35:00,44629;452;42833;78959;5180;5849;70719;5728;571...,Other specified hypersensitivity angiitis;Port...,Furosemide;Glucagon;Metoprolol Tartrate;Insuli...,8611;3723;8853;8856,Closed biopsy of skin and subcutaneous tissue;...
72237,190677,2,2140-05-09 14:05:00,2140-05-16 11:28:00,0389;78552;5722;2869;2639;44629;70710;5849;276...,Unspecified septicemia;Septic shock;Hepatic en...,Insulin;Lactulose;Aspirin;Lidocaine Jelly 2% (...,3897,Central venous catheter placement with guidance
